In [6]:
#Manipulating Prosper Loan Data to Recover Prepayment and Default Information
import pandas as pd

file_path = 'prosperLoanData.csv'
df = pd.read_csv(file_path,engine='python', on_bad_lines='skip')
df.head()

#Clean up time objects
df['ListingCreationDate'] = pd.to_datetime(df['ListingCreationDate'],  format='mixed').dt.date
df['ClosedDate'] = pd.to_datetime(df['ClosedDate'],  format='mixed').dt.date

#Eliminate loans that were cancelled, as these are not relevant for our analysis of prepayment and default risk.
df = df[~df['LoanStatus'].isin(['Cancelled'])]

#Create a column to determine if a loan was prepaid
df['CompletedLoans'] = df['LoanStatus'].isin(['Completed'])
df['Prepayment'] = df['CompletedLoans'] & ((pd.to_datetime(df['ClosedDate']) - pd.to_datetime(df['LoanOriginationDate'])).dt.days / 30 < df['Term']-1)
#Create a column to indicate if a loan was prepaid
df['IsPrepaid'] = df['Prepayment'].astype(int)

#If a loan was prepaid, change the loan status to 'Prepaid'
df.loc[df['IsPrepaid'] == 1, 'LoanStatus'] = 'Prepaid'

#Create a new column that codifies the loan status.
#Assign 0 for Current Loans, 1 for Completed Loans, 2 for Defaulted Loans, and 3 for Prepaid Loans.
def categorize_loan_status(status):
    if status in ['Current', 'Past Due (1-15 days)', 'Past Due (31-60 days)', 'Past Due (16-30 days)',  'Past Due (61-90 days)', 'Past Due (91-120 days)', 'Past Due (>120 days)', 'Past Due (181-360 days)', 'Past Due (16-30 days)', 'Past Due (31-60 days)', 'Past Due ( days)', 'Past Due (Over 720 days)', 'FinalPaymentInProgress']:
        return 0
    elif status in ['Completed', 'FinalPaymentInProgress']:
        return 1
    elif status in ['Defaulted', 'Chargedoff']:
        return 2
    elif status in ['Prepaid']:
        return 3
    else:
        return -1  # For any other statuses not covered above

df['LoanStatusCategory'] = df['LoanStatus'].apply(categorize_loan_status)
#Create a column that indicates how many months it took to repay or default a loan
df['MonthsToRepayOrDefault'] = ((pd.to_datetime(df['ClosedDate']) - pd.to_datetime(df['LoanOriginationDate'])).dt.days / 30).fillna(-1)
#Round MonthsTORepayOrDefault to nearest integer
df['MonthsToRepayOrDefault'] = df['MonthsToRepayOrDefault'].round()


#Look at the loan that defaulted with a negative MonthsToRepayOrDefault
df[(df['MonthsToRepayOrDefault'] < 0) & (df['LoanStatusCategory'] == 2)]
#throw this single data point out as a missed input
df = df.drop(df[(df['MonthsToRepayOrDefault'] < 0) & (df['LoanStatusCategory'] == 2)].index)
#Check that the data point has been removed
df[(df['MonthsToRepayOrDefault'] < 0) & (df['LoanStatusCategory'] == 2)]


#Check that only current loans have a negative MonthsToRepayOrDefault
df[df['MonthsToRepayOrDefault'] < 0]['LoanStatus'].value_counts()

#Value count for each event
df['LoanStatusCategory'].value_counts()



,count
LoanStatusCategory,
0,57080
3,26674
2,16516
1,10257


In [7]:
#We want to make sure that there is no data on months for current/prepaid/defaulted loans past their last observed month.
import pandas as pd
import numpy as np

as_of_date = pd.to_datetime("2014-03-10")
df["ListingCreationDate"] = pd.to_datetime(df["ListingCreationDate"])

df["MonthsFromOrigToAsOf"] = ((as_of_date.year - df["ListingCreationDate"].dt.year) * 12 +
                              (as_of_date.month - df["ListingCreationDate"].dt.month))

df["LastObservedMonth"] = np.where(
    df["LoanStatusCategory"].isin([2, 3]),
    df["MonthsToRepayOrDefault"],
    df["MonthsFromOrigToAsOf"]
)

for month in range(1, 37):
    col_name = f"Month_{month}"
    event_indicator = np.zeros(len(df), dtype=float)
    prepay_mask = (df["LoanStatusCategory"] == 3) & (df["MonthsToRepayOrDefault"] == month)
    default_mask = (df["LoanStatusCategory"] == 2) & (df["MonthsToRepayOrDefault"] == month)

    event_indicator[prepay_mask] = 1
    event_indicator[default_mask] = 2

    # Now NaN assignment works
    censored_mask = df["LastObservedMonth"] < month
    event_indicator[censored_mask] = np.nan

    df[col_name] = event_indicator

df.head()



,ListingKey,ListingNumber,ListingCreationDate,CreditGrade,Term,LoanStatus,ClosedDate,BorrowerAPR,BorrowerRate,LenderYield,...,Month_27,Month_28,Month_29,Month_30,Month_31,Month_32,Month_33,Month_34,Month_35,Month_36
0,1021339766868145413AB3B,193129,2007-08-26,C,36,Prepaid,2009-08-14,0.16516,0.1580,0.1380,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10273602499503308B223C1,1209647,2014-02-27,NaN,36,Current,NaT,0.12016,0.0920,0.0820,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0EE9337825851032864889A,81716,2007-01-05,HR,36,Completed,2009-12-17,0.28269,0.2750,0.2400,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0EF5356002482715299901A,658116,2012-10-22,NaN,36,Current,NaT,0.12528,0.0974,0.0874,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0F023589499656230C5E3E2,909464,2013-09-14,NaN,36,Current,NaT,0.24614,0.2085,0.1985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
fed_df = pd.read_csv("FEDFUNDS (1).csv")
fed_df["observation_date"] = pd.to_datetime(fed_df["observation_date"])
fed_df = fed_df.sort_values("observation_date").drop_duplicates("observation_date").reset_index(drop=True)

print("FED data preview:")
print(fed_df.head())

# 3-class: 0=current/completed, 1=default, 2=prepay
df["Event36"] = np.where(
    df["LoanStatusCategory"].isin([0, 1]),  # current or completed
    0,
    np.where(
        df["LoanStatusCategory"] == 2,      # defaulted/chargedoff
        1,
        2                                   # prepaid
    )
)


FED data preview:
  observation_date  FEDFUNDS
0       1954-07-01      0.80
1       1954-08-01      1.22
2       1954-09-01      1.07
3       1954-10-01      0.85
4       1954-11-01      0.83


In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score
import joblib

# ===== 1. FED PREP =====
fed_df = pd.read_csv("FEDFUNDS (1).csv")
fed_df["observation_date"] = pd.to_datetime(fed_df["observation_date"])
#Calculate fed rate at loan origin for each loan.
fed_monthly = fed_df.set_index('observation_date').resample('MS').interpolate('linear').ffill()
fed_dict = fed_monthly['FEDFUNDS'].to_dict()
print(f"FED ready: {len(fed_dict)} months")

# ===== 2. MONTHLY DATAFRAME =====
#Add month-by-month data for each loan, including the borrower rate, FED rate, spread/incentive info, and previous 3 month FED change.

monthly_rows = []
zero_apr_loans = 0

for idx, loan in df.iterrows():
    borrower_apr = loan['BorrowerRate']
    #Throw out "weird" loans with zero or negative APR
    if borrower_apr <= 0:
        zero_apr_loans += 1
        continue

    #Only add info on last observed month
    last_month = int(loan['LastObservedMonth'])
    #Gather data on each month since loan origination
    for month in range(1, min(37, last_month + 1)):
        month_date = loan['ListingCreationDate'] + pd.DateOffset(months=month-1)
        #retrieve fed rate for this month
        fed_now = fed_dict.get(month_date, 0.02)

        # Look at the FED rate 3 months prior (capturing some volatility)
        fed_3mo_ago_date = month_date - pd.DateOffset(months=3)
        fed_3mo_ago = fed_dict.get(fed_3mo_ago_date, fed_now)

        #Calculate the borrower's incentive to refinance this month
        apr_fed_spread = borrower_apr - fed_now
        #Compare with 3 months ago
        apr_fed_spread_change = (borrower_apr - fed_3mo_ago) - apr_fed_spread
        #Calculate fed change
        fed_change_3m = fed_now - fed_3mo_ago

        #Add all info into month-by-month dataframe.
        monthly_rows.append({
            'loan_id': idx,
            'cohort_year': loan['ListingCreationDate'].year,
            'month_since_orig': month,
            'status_next': loan[f'Month_{month}'],
            'apr_fed_spread': apr_fed_spread,
            'apr_fed_spread_change_3m': apr_fed_spread_change,
            'apr_fed_spread_pct': apr_fed_spread / borrower_apr,
            'fed_change_3m': fed_change_3m,
            'fed_level': fed_now,
            'borrower_apr': borrower_apr,
            'loan_amount_log': np.log1p(loan.get('LoanOriginalAmount', 10000))
        })

monthly_df = pd.DataFrame(monthly_rows)
print(f"Dataset: {len(monthly_df):,} loan-months ({zero_apr_loans} zero-APR skipped)")
print("Events:", dict(monthly_df['status_next'].value_counts()))


FED ready: 787 months
Dataset: 1,612,254 loan-months (7 zero-APR skipped)
Events: {0.0: np.int64(1570140), 1.0: np.int64(26266), 2.0: np.int64(15848)}


In [10]:
selected_features = ['month_since_orig', 'apr_fed_spread', 'fed_level', 'borrower_apr', 'loan_amount_log']

In [11]:
# Combined data: predict BOTH default AND prepay hazards
# Temporal split: train/valid/test by cohort_year
#Smaller train set to prevent overfit in the DL models

train_mask = monthly_df['cohort_year'] <= 2010
valid_mask = monthly_df['cohort_year'] == 2011
test_mask  = monthly_df['cohort_year'] >= 2012

# Multi-task targets
y_def_train = (monthly_df[train_mask]['status_next'] == 1).astype(int)
y_prepay_train = (monthly_df[train_mask]['status_next'] == 2).astype(int)

y_def_valid = (monthly_df[valid_mask]['status_next'] == 1).astype(int)
y_prepay_valid = (monthly_df[valid_mask]['status_next'] == 2).astype(int)

y_def_test = (monthly_df[test_mask]['status_next'] == 1).astype(int)
y_prepay_test = (monthly_df[test_mask]['status_next'] == 2).astype(int)

X_train = monthly_df[train_mask][selected_features].values
X_valid = monthly_df[valid_mask][selected_features].values
X_test  = monthly_df[test_mask][selected_features].values


In [12]:
#Drop the non-observed months
risk_df = monthly_df[monthly_df['status_next'].notna()]
prepay_data = risk_df[risk_df['status_next'].isin([0,2])]

#Train Random Forests
# Train finals on train data only (2005-2010)
train_mask = risk_df['cohort_year'] <= 2010

def_data_final = risk_df[(risk_df['status_next'].isin([0, 1])) & train_mask]
prepay_data_final = risk_df[(risk_df['status_next'].isin([0, 2])) & train_mask]

print(f"Final train cohorts - Default: {sorted(def_data_final['cohort_year'].unique())}")
print(f"Final train cohorts - Prepay:  {sorted(prepay_data_final['cohort_year'].unique())}")

# Default model
rf_default_final = RandomForestClassifier(
    n_estimators=500,
    class_weight={0: 1, 1: 25},
    max_depth=8,
    min_samples_leaf=100,
    n_jobs=-1,
    random_state=42,
    oob_score=True,
)

rf_default_final.fit(
    def_data_final[selected_features],
    (def_data_final['status_next'] == 1).astype(int),
)

# Prepay model
rf_prepay_final = RandomForestClassifier(
    n_estimators=500,
    class_weight={0: 1, 1: 6},
    max_depth=8,
    min_samples_leaf=100,
    n_jobs=-1,
    random_state=42,
    oob_score=True,
)

rf_prepay_final.fit(
    prepay_data_final[selected_features],
    (prepay_data_final['status_next'] == 2).astype(int),
)

Final train cohorts - Default: [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010)]
Final train cohorts - Prepay:  [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010)]


RandomForestClassifier(class_weight={0: 1, 1: 6}, max_depth=8,
                       min_samples_leaf=100, n_estimators=500, n_jobs=-1,
                       oob_score=True, random_state=42)

In [15]:
import joblib

# Save BOTH models
joblib.dump(rf_default_final, 'rf_default_model.pkl')
joblib.dump(rf_prepay_final, 'rf_prepay_model.pkl')
joblib.dump(selected_features, 'rf_features.pkl')  # Feature names

print("✅ Models saved!")
from google.colab import files
files.download('rf_default_model.pkl')
files.download('rf_prepay_model.pkl')
files.download('rf_features.pkl')


✅ Models saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>